In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, to_date

spark = SparkSession.builder \
    .appName("WebServerLogsAnalysis") \
    .getOrCreate()

df = spark.read.csv("web_server_logs.csv", header=True, inferSchema=True)

print("Top 10 active IP addresses:")
df.groupBy("ip") \
  .agg(count("*").alias("request_count")) \
  .orderBy(col("request_count").desc()) \
  .limit(10) \
  .show()

print("Request count by HTTP method:")
df.groupBy("method") \
  .agg(count("*").alias("method_count")) \
  .orderBy(col("method_count").desc()) \
  .show()

count_404 = df.filter(col("response_code") == 404).count()
print(f"Number of 404 response codes: {count_404}")

print("Total response size by day:")
df.withColumn("date", to_date(col("timestamp"))) \
  .groupBy("date") \
  .agg(spark_sum("response_size").alias("total_response_size")) \
  .orderBy("date") \
  .show(365, truncate=False)

spark.stop()